# 🎬 Reverse Prompt Engineering for AI Educational Videos
**向上溯源反推系统 - Google Colab T4 运行版**

This notebook runs the full pipeline on a T4 GPU:
1. Clone & install → 2. Download videos → 3. Reverse-engineer prompts → 4. Pattern analysis

In [ ]:
# ============================================================
# CELL 1: Check GPU
# ============================================================
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - Please enable T4 in Runtime settings"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB' if torch.cuda.is_available() else '')

In [ ]:
# ============================================================
# CELL 2: Clone Repository
# ============================================================
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # <-- UPDATE THIS
REPO_DIR = '/content/reverse_prompt'

if os.path.exists(REPO_DIR):
    print('Repo exists, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ============================================================
# CELL 3: Install Dependencies
# ============================================================
!pip install -r requirements.txt -q
print('✅ Dependencies installed')

In [ ]:
# ============================================================
# CELL 4: (Optional) Mount Google Drive to cache model weights
# Saves ~7GB download time on subsequent runs
# ============================================================
USE_GDRIVE = True  # Set False to skip

if USE_GDRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Point HuggingFace cache to Google Drive
    import os
    CACHE_DIR = '/content/drive/MyDrive/hf_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = CACHE_DIR
    print(f'✅ HuggingFace cache → Google Drive: {CACHE_DIR}')
else:
    print('Skipped Google Drive mount. Models will be re-downloaded each session.')

In [ ]:
# ============================================================
# CELL 5: Download Video Dataset
# ============================================================
# Downloads curated golden + regular educational videos
# golden: cinematic/AIGC quality (3Blue1Brown, Kurzgesagt etc.)
# regular: standard classroom/screencast style

MAX_VIDEOS_PER_GROUP = 5  # Adjust as needed (more = longer runtime)

!python main.py --download --max {MAX_VIDEOS_PER_GROUP}
print('✅ Download complete')

In [ ]:
# ============================================================
# CELL 6: Run Pipeline - Golden Videos
# ============================================================
# This will:
#   1. Extract keyframes from each video
#   2. Load InternVL2-2B (~4GB) and analyze frames
#   3. Run 3 agents with Qwen2.5-1.5B (~3GB)
#   4. Synthesize and save reversed prompts

!python main.py --batch data/raw_videos/golden/ --group golden --output results/golden/ --lang en

In [ ]:
# ============================================================
# CELL 7: Run Pipeline - Regular Videos
# ============================================================
!python main.py --batch data/raw_videos/regular/ --group regular --output results/regular/ --lang en

In [ ]:
# ============================================================
# CELL 8: Pattern Analysis (CPU only, no GPU needed)
# ============================================================
# Merge golden and regular results into one directory for analysis
import os, shutil
os.makedirs('results/all', exist_ok=True)
for src_dir in ['results/golden', 'results/regular']:
    for f in os.listdir(src_dir) if os.path.exists(src_dir) else []:
        if f.endswith('.json'):
            shutil.copy(os.path.join(src_dir, f), 'results/all/')

!python main.py --analyze --results results/all/

In [ ]:
# ============================================================
# CELL 9: View Results
# ============================================================
import json

# Show the pattern report
report_path = 'results/all/pattern_report.json'
if os.path.exists(report_path):
    with open(report_path) as f:
        report = json.load(f)
    
    print('=== 🏆 Golden Prompt Keywords ===')
    print(', '.join(report['golden_keywords'][:15]))
    
    print('\n=== 📚 Regular Prompt Keywords ===')
    print(', '.join(report['regular_keywords'][:15]))
    
    print('\n=== 📊 Stats ===')
    print(json.dumps(report['stats'], indent=2))
    
    print('\n=== 💡 Summary ===')
    print(report['summary'])
else:
    print('No report found. Run Cell 8 first.')

In [ ]:
# ============================================================
# CELL 10: GPU Memory Check
# ============================================================
import torch
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU Memory: {allocated:.1f}GB used / {reserved:.1f}GB reserved / {total:.1f}GB total')
    print(f'Available: {total - reserved:.1f}GB')

In [ ]:
# ============================================================
# CELL 11: (Optional) Download Results to Local
# ============================================================
from google.colab import files
import zipfile

# Zip results directory
with zipfile.ZipFile('results.zip', 'w') as zf:
    for root, dirs, filenames in os.walk('results'):
        for filename in filenames:
            if filename.endswith('.json'):
                zf.write(os.path.join(root, filename))

files.download('results.zip')
print('✅ results.zip downloaded')